In [1]:
#Setup and Load Data

In [ ]:
import pandas as pd

In [2]:
df = pd.read_csv("SuperStoreOrders.csv", encoding="utf-8-sig")

In [6]:
df.dtypes

order_id              str
order_date            str
ship_date             str
ship_mode             str
customer_name         str
segment               str
state                 str
country               str
market                str
region                str
product_id            str
category              str
sub_category          str
product_name          str
sales                 str
quantity            int64
discount          float64
profit            float64
shipping_cost     float64
order_priority        str
year                int64
dtype: object

In [ ]:
#Data Cleaning

In [7]:
df["sales"] = pd.to_numeric(
    df["sales"].str.strip().str.replace(",", "", regex=False),
    errors="coerce"
).astype("float64")

print("Missing sales:", df["sales"].isna().sum())

Missing sales: 0


In [8]:
df["sales"] = df["sales"].astype("int64")
print(df["sales"].dtype)

int64


In [9]:
for col in ["order_date","ship_date"]:
    df[col] = df[col].str.replace("-", "/", regex = False)

In [10]:
for col in ["order_date", "ship_date"]:
    df[col] = pd.to_datetime(df[col], format = "%d/%m/%Y", errors = "coerce")

In [11]:
df["order_id"] = df["order_id"].str.strip().str.upper()
print("Blank order IDs:", (df["order_id"] == "").sum() + df["order_id"].isna().sum())

Blank order IDs: 0


In [12]:
df["year"] = df["year"].astype("int64")

In [ ]:
#Data Valiadtion

In [13]:
print("Ship before order:", (df["ship_date"] < df["order_date"]).sum())

Ship before order: 0


In [14]:
print("Negative sales:", (df["sales"] < 0).sum())
print("Negative quantity:", (df["quantity"] < 0).sum())
print("Discount outside 0-1:", ((df["discount"] < 0) | (df["discount"] > 1)).sum())

Negative sales: 0
Negative quantity: 0
Discount outside 0-1: 0


In [15]:
print(df[["sales", "quantity", "discount", "profit", "shipping_cost"]].describe())
print(df.isna().sum())

              sales      quantity      discount        profit  shipping_cost
count  51290.000000  51290.000000  51290.000000  51290.000000   51290.000000
mean     246.498440      3.476545      0.142908     28.641740      26.375915
std      487.567175      2.278766      0.212280    174.424113      57.296804
min        0.000000      1.000000      0.000000  -6599.978000       0.000000
25%       31.000000      2.000000      0.000000      0.000000       2.610000
50%       85.000000      3.000000      0.000000      9.240000       7.790000
75%      251.000000      5.000000      0.200000     36.810000      24.450000
max    22638.000000     14.000000      0.850000   8399.976000     933.570000
order_id          0
order_date        0
ship_date         0
ship_mode         0
customer_name     0
segment           0
state             0
country           0
market            0
region            0
product_id        0
category          0
sub_category      0
product_name      0
sales             0
quantit

In [16]:
print("Exact duplicates:", df.duplicated().sum())

Exact duplicates: 0


In [17]:
dupe_rows = df[df.duplicated(["order_id", "product_id"], keep=False)]

In [18]:
len(dupe_rows), dupe_rows["order_id"].nunique()

(76, 38)

In [19]:
(df["order_date"].dt.year == df["year"]).all()

np.True_

In [ ]:
#Feature Engineering

In [20]:
df["delivery_days"] = (df["ship_date"] - df["order_date"]).dt.days
print(df["delivery_days"].min(), df["delivery_days"].max())

0 7


In [21]:
df["order_month"] = df["order_date"].dt.month
df["order_quarter"] = df["order_date"].dt.quarter

In [22]:
df["profit_margin"] = df["profit"] / df["sales"]
df.loc[df["sales"] == 0, "profit_margin"] = float("nan")

In [ ]:
#Final Check and Save

In [23]:
print(df.shape)
print(df["order_date"].isna().sum(), df["ship_date"].isna().sum())
print(df["sales"].dtype, df["sales"].sum())

(51290, 25)
0 0
int64 12642905


In [24]:
df.to_csv("superstore_clean.csv", index=False)